# Lecture 3: Algerian Forest Fires - Model Training Workflow

### Short, cell-by-cell, self-study notes

This project uses the Algerian Forest Fires dataset to predict **Temperature** from fire-weather measurements. It connects cleaning, EDA, model training, and saving a model for later use.

**Main idea:** A reliable model workflow is clean data → safe split → preprocessing → model comparison → final test → saved pipeline.

## 1. Project files and goal

The extracted practical folder contains:

- A raw Algerian forest-fire CSV.
- A cleaned CSV.
- The original EDA and model-training notebooks.
- A saved scaler and Ridge model from the original workflow.

This simplified lesson uses the cleaned CSV. The target is Temperature, a numeric continuous value, so this is a regression problem.

## 2. Cell 1: project workflow diagram

The diagram shows the order of work. The final test data stays separate until we are ready to compare trained models.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

steps = ["Clean CSV", "Choose X and y", "Train/test split", "Scale in pipeline", "Fit models", "Test once", "Save pipeline"]
fig, ax = plt.subplots(figsize=(15, 3))
ax.set_xlim(0, 14)
ax.set_ylim(0, 2)
ax.axis("off")

for index, step in enumerate(steps):
    x = 0.15 + index * 1.95
    box = FancyBboxPatch((x, 0.75), 1.55, 0.55, boxstyle="round,pad=0.04",
                         facecolor="#d9eaf7", edgecolor="#2f5d8a")
    ax.add_patch(box)
    ax.text(x + 0.775, 1.03, step, ha="center", va="center", fontsize=9, weight="bold")
    if index < len(steps) - 1:
        ax.add_patch(FancyArrowPatch((x + 1.58, 1.03), (x + 1.88, 1.03),
                                     arrowstyle="->", mutation_scale=14, color="#2f5d8a"))

ax.set_title("Forest-fire regression project workflow", weight="bold")
plt.show()

## 3. Cell 2: load the cleaned data

The original notebook performs detailed cleaning on the raw file. This lesson reads the already cleaned CSV so we can focus on understandable model training.

The Classes text column is cleaned again with strip and lowercase before encoding. This prevents spaces from creating accidental extra categories.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns

data_path = Path("Model Training Practicals") / "Algerian_forest_fires_cleaned_dataset.csv"
df = pd.read_csv(data_path)
df["Classes"] = df["Classes"].str.strip().str.lower()
df["Classes"] = df["Classes"].map({"not fire": 0, "fire": 1})

if df["Classes"].isna().any():
    raise ValueError("Unexpected class label found. Check the label-cleaning step.")

print("Dataset shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
df.head()

## 4. Cell 3: choose the target and input features

We predict Temperature. The date fields are removed for this first model, matching the transcript. Classes and Region are kept as numeric inputs after cleaning.

In a real project, do not drop features only by habit. Keep or remove them based on domain knowledge, validation results, and the business question.

In [ ]:
target_column = "Temperature"
columns_to_remove = ["day", "month", "year", target_column]

X = df.drop(columns=columns_to_remove)
y = df[target_column]

print("Input features:", list(X.columns))
print("Target:", target_column)
X.head()

## 5. Cell 4: inspect relationships

The heatmap helps us understand linear relationships between input features and Temperature. Strongly related input features may cause multicollinearity, which can make ordinary Linear Regression coefficients unstable.

Ridge and Elastic Net can help by shrinking coefficients, but correlation alone should not decide which features are removed.

In [ ]:
correlation_for_plot = pd.concat([X, y], axis=1).corr()
plt.figure(figsize=(11, 8))
sns.heatmap(correlation_for_plot, cmap="coolwarm", center=0, square=True, linewidths=0.3)
plt.title("Feature and Temperature correlations", weight="bold")
plt.show()

correlation_for_plot[target_column].sort_values(ascending=False).round(3)

## 6. Cell 5: split before scaling

The test set is held out before fitting any preprocessing or models. It gives an honest final check on unseen data.

The pipelines in the next cell learn scaling from the training rows only. This is safer than manually scaling all rows before the split.

In [ ]:
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Training data:", X_train.shape)
print("Final test data:", X_test.shape)

## 7. Cell 6: train and compare four models

Every model uses the same data split and the same StandardScaler pipeline. This makes the comparison fair.

- Linear Regression is the baseline.
- Ridge uses an L2 penalty.
- Lasso uses an L1 penalty and may remove features.
- Elastic Net mixes both penalties.

The penalty settings below are examples. Cross-validation should choose final settings in a production project.

In [ ]:
models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=20_000)),
    "Elastic Net": make_pipeline(StandardScaler(), ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=20_000)),
}

results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    train_prediction = model.predict(X_train)
    test_prediction = model.predict(X_test)
    predictions[name] = test_prediction
    results.append({
        "model": name,
        "train_R2": r2_score(y_train, train_prediction),
        "test_R2": r2_score(y_test, test_prediction),
        "test_MAE": mean_absolute_error(y_test, test_prediction),
    })

results_table = pd.DataFrame(results).set_index("model").round(3)
results_table

### How to read the comparison

- Higher test R² is generally better.
- Lower test MAE is generally better.
- A large difference between train R² and test R² can be a warning sign for overfitting.
- Do not choose a final model from one test split alone. Use cross-validation for more reliable selection.

## 8. Cell 7: visual test comparison

Each dot is a test example. The dashed diagonal line means perfect prediction. The closer the dots are to the line, the more accurate the predictions are for those rows.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=True, sharey=True)
all_values = np.concatenate([y_test.to_numpy(), *predictions.values()])
line_min, line_max = all_values.min(), all_values.max()

for axis, (name, prediction) in zip(axes.ravel(), predictions.items()):
    axis.scatter(y_test, prediction, alpha=0.7, color="#457b9d")
    axis.plot([line_min, line_max], [line_min, line_max], "--", color="#e76f51")
    axis.set_title(name)
    axis.set_xlabel("Actual Temperature")
    axis.set_ylabel("Predicted Temperature")
    axis.grid(alpha=0.2)

fig.suptitle("Test predictions: closer to the diagonal is better", weight="bold")
fig.tight_layout()
plt.show()

## 9. Cell 8: save one complete pipeline

The original ZIP stores ridge.pkl and scaler.pkl as separate files. That works, but saving a single fitted pipeline is simpler and safer because it keeps preprocessing and the model together.

This cell saves the fitted Ridge pipeline. It can later be loaded and given raw feature columns in the same order.

In [ ]:
import pickle

ridge_pipeline = models["Ridge"]
model_path = "temperature_ridge_pipeline.pkl"

with open(model_path, "wb") as file:
    pickle.dump(ridge_pipeline, file)

with open(model_path, "rb") as file:
    loaded_pipeline = pickle.load(file)

example_prediction = loaded_pipeline.predict(X_test.iloc[[0]])[0]
print(f"Loaded pipeline prediction for the first test row: {example_prediction:.2f}")

## 10. Final revision card

- The Algerian Forest Fires project has two regions and weather/fire-index features.
- In this regression example, Temperature is the numeric target.
- Clean text labels before encoding them as numbers.
- Split before scaling or training to avoid data leakage.
- A pipeline keeps scaling and prediction together in the correct order.
- Compare models fairly with the same training and test sets.
- Use test MAE and R², then use cross-validation for final hyperparameter selection.
- Save the whole pipeline so deployment applies the same preprocessing.

### One-line interview answer

**For a deployment-ready regression workflow, I clean the data, split before preprocessing, train models in pipelines, evaluate untouched test data, and serialize the complete fitted pipeline.**